# Multilingual Emotion Model Training: GoEmotions + ArPanEmo

This notebook fine-tunes a multilingual BERT encoder on two emotion datasets and harmonizes them into one shared 7-way emotion space:

- `joy`
- `sadness`
- `anger`
- `fear`
- `surprise`
- `disgust`
- `neutral`

Why this version exists:

- GoEmotions is rich but English-only.
- ArPanEmo adds Arabic coverage.
- A shared label space makes the bilingual model much more stable than forcing 28 fine-grained GoEmotions labels onto both corpora.
- The saved model will later be copied into `models/fine_tuned_bert/` and uploaded to Google Drive from Colab.

This notebook is intentionally more training-heavy than the earlier 20-minute baseline. On a Colab T4, expect a longer run because we use the full English corpus, smaller batches, weighted loss, and more epochs.


## 1. Install and Import Dependencies

If you are running this notebook in a fresh Colab environment, you can uncomment the install line once.
We keep the imports explicit so the notebook stays readable and the training pipeline is easy to follow.


In [ ]:
# If you are running this notebook in a fresh environment, uncomment the next line.
# !pip install -q datasets transformers accelerate scikit-learn seaborn

from __future__ import annotations

import json
import os
import re
import shutil
from collections import Counter
from datetime import datetime, timezone
from inspect import signature
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)
print('Dependencies imported successfully.')


## 2. Configuration

We keep the configuration in one place so the notebook is easy to tune later.

Important choices in this version:

- We use a multilingual BERT encoder that is stronger for social text.
- We train on a shared 7-class emotion set.
- We use smaller batches and more epochs so the run is slower but usually more meaningful.


In [ ]:
BASE_MODEL_NAME = 'bert-base-multilingual-uncased'
ENGLISH_DATASET_NAME = 'mrm8488/goemotions'
ARABIC_DATASET_NAME = 'hanaearg/arpanemo-emotion-dataset-v4-safe-split'

OUTPUT_DIR_PATH = Path('models/fine_tuned_bert')
MAX_LENGTH = 128
MAX_ENGLISH_SAMPLES = None  # Use the full GoEmotions corpus for better coverage.
ARABIC_TRAIN_REPEAT_FACTOR = 5
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
NUM_EPOCHS = 4
LEARNING_RATE = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRADIENT_ACCUMULATION_STEPS = 2
TEST_SIZE = 0.2
RANDOM_SEED = 42

set_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# GoEmotions raw label columns in the Hugging Face dataset schema.
GOEMOTIONS_RAW_LABELS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
    'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
    'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse',
    'sadness', 'surprise', 'neutral',
]

# Shared bilingual label space. This is the actual model output space.
SHARED_EMOTION_LABELS = ['joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust', 'neutral']
LABEL2ID = {label: idx for idx, label in enumerate(SHARED_EMOTION_LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

# Map the fine-grained GoEmotions labels into the shared 7-way space.
GOEMOTIONS_TO_SHARED = {label: 'joy' for label in ['admiration', 'amusement', 'approval', 'caring', 'desire', 'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief']}
GOEMOTIONS_TO_SHARED.update({label: 'sadness' for label in ['disappointment', 'embarrassment', 'grief', 'remorse', 'sadness']})
GOEMOTIONS_TO_SHARED.update({label: 'anger' for label in ['anger', 'annoyance', 'disapproval']})
GOEMOTIONS_TO_SHARED.update({label: 'fear' for label in ['fear', 'nervousness']})
GOEMOTIONS_TO_SHARED.update({label: 'surprise' for label in ['confusion', 'curiosity', 'realization', 'surprise']})
GOEMOTIONS_TO_SHARED.update({label: 'disgust' for label in ['disgust']})
GOEMOTIONS_TO_SHARED.update({label: 'neutral' for label in ['neutral']})

ARABIC_TO_SHARED = {
    'happiness': 'joy',
    'sadness': 'sadness',
    'anger': 'anger',
    'fear': 'fear',
    'surprise': 'surprise',
    'disgust': 'disgust',
    'neutral': 'neutral',
}

# Used to break ties when an English GoEmotions row maps to multiple shared labels.
COARSE_LABEL_PRIORITY = ['anger', 'disgust', 'fear', 'sadness', 'surprise', 'joy', 'neutral']
POSITIVE_EMOTIONS = ['joy']
NEGATIVE_EMOTIONS = ['sadness', 'anger', 'fear', 'disgust']
NEUTRAL_EMOTIONS = ['surprise', 'neutral']

print('Device detected:', DEVICE)
print('Output directory:', OUTPUT_DIR_PATH.resolve())
print('English dataset:', ENGLISH_DATASET_NAME)
print('Arabic dataset:', ARABIC_DATASET_NAME)
print('Shared label space:', SHARED_EMOTION_LABELS)
print('Number of target labels:', len(SHARED_EMOTION_LABELS))


## 3. Load Both Datasets

We load the datasets first so we can inspect their schema before we transform anything.

Expected output:

- English GoEmotions train split
- Arabic ArPanEmo train/validation/test splits
- sample rows from both datasets


In [ ]:

english_ds = load_dataset(ENGLISH_DATASET_NAME, split='train')
arabic_ds = load_dataset(ARABIC_DATASET_NAME)

print('English dataset loaded:')
print(english_ds)
print('Arabic dataset loaded:')
print(arabic_ds)

print('Sample English rows:')
display(english_ds.select(range(3)).to_pandas())

print('Sample Arabic rows:')
display(arabic_ds['train'].select(range(3)).to_pandas())

print('Arabic split sizes:')
for split_name in arabic_ds.keys():
    print('-', split_name, len(arabic_ds[split_name]))


## 4. Helper Functions

The key idea in this section is **label harmonization**.

We do not want two incompatible label systems. Instead, we convert both corpora into the same shared 7-way emotion space.

That gives us:

- a smaller and more stable label space
- easier multilingual training
- cleaner evaluation and inference later


In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
WHITESPACE_RE = re.compile(r'\s+')


def clean_text(text):
    # Minimal cleaning only. We remove URLs and collapse whitespace, but we do not over-normalize Arabic.
    if text is None:
        return ''
    text = str(text)
    text = URL_RE.sub(' ', text)
    text = WHITESPACE_RE.sub(' ', text)
    return text.strip()


def map_goemotions_row_to_shared(row):
    # GoEmotions is multi-label. We keep one shared label per row for the first training version.
    # If multiple emotions are present, we map them to the shared 7-way space and pick the most frequent coarse label.
    positive_positions = np.flatnonzero(row)
    if len(positive_positions) == 0:
        return 'neutral'

    mapped_labels = [GOEMOTIONS_TO_SHARED[GOEMOTIONS_RAW_LABELS[int(pos)]] for pos in positive_positions]
    counts = Counter(mapped_labels)
    best_count = max(counts.values())
    candidates = [label for label, count in counts.items() if count == best_count]

    for label in COARSE_LABEL_PRIORITY:
        if label in candidates:
            return label

    return candidates[0]


def build_english_dataframe(dataset):
    keep_columns = ['text', 'example_very_unclear'] + GOEMOTIONS_RAW_LABELS
    drop_columns = [col for col in dataset.column_names if col not in keep_columns]
    small_ds = dataset.remove_columns(drop_columns)
    df = small_ds.to_pandas()

    # Remove ambiguous rows.
    df = df[df['example_very_unclear'] == False].copy()  # noqa: E712

    # Use the full English corpus unless we intentionally cap it for a quick experiment.
    if MAX_ENGLISH_SAMPLES is not None and len(df) > MAX_ENGLISH_SAMPLES:
        df = df.sample(n=MAX_ENGLISH_SAMPLES, random_state=RANDOM_SEED).copy()

    # Convert the fine-grained GoEmotions annotations into the shared bilingual label space.
    label_matrix = df[GOEMOTIONS_RAW_LABELS].to_numpy(dtype=np.int8)
    shared_labels = [map_goemotions_row_to_shared(row) for row in label_matrix]

    df['label_name'] = shared_labels
    df['language'] = 'en'
    df['source_dataset'] = ENGLISH_DATASET_NAME
    df['text'] = df['text'].map(clean_text)
    return df[['text', 'label_name', 'language', 'source_dataset']].reset_index(drop=True)


def build_arabic_dataframe(dataset_dict):
    frames = []
    for split_name in dataset_dict.keys():
        split_df = dataset_dict[split_name].to_pandas()
        split_df['label_name'] = split_df['output'].map(ARABIC_TO_SHARED)
        split_df['language'] = 'ar'
        split_df['source_dataset'] = ARABIC_DATASET_NAME
        split_df['text'] = split_df['input'].map(clean_text)
        split_df = split_df.dropna(subset=['label_name'])
        frames.append(split_df[['text', 'label_name', 'language', 'source_dataset']])

    return pd.concat(frames, ignore_index=True)


def sentiment_from_emotion(label_name):
    if label_name in POSITIVE_EMOTIONS:
        return 'positive'
    if label_name in NEGATIVE_EMOTIONS:
        return 'negative'
    return 'neutral'


def table_counts(series):
    return series.value_counts().rename_axis(series.name).reset_index(name='count')


## 5. Build the Combined Multilingual Dataset

We now put both corpora into the same schema:

- `text`
- `label_name`
- `language`
- `source_dataset`

At this point both English and Arabic are living in the same shared 7-way emotion space, which is the main design decision that makes the bilingual training work.


In [ ]:
english_df = build_english_dataframe(english_ds)
arabic_df = build_arabic_dataframe(arabic_ds)

english_df['label_id'] = english_df['label_name'].map(LABEL2ID)
arabic_df['label_id'] = arabic_df['label_name'].map(LABEL2ID)

combined_df = pd.concat([english_df, arabic_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print('English rows:', len(english_df))
print('Arabic rows:', len(arabic_df))
print('Combined rows:', len(combined_df))

print('Language distribution:')
display(table_counts(combined_df['language']))

print('Shared label distribution:')
display(table_counts(combined_df['label_name']))

print('Sample combined rows:')
display(combined_df.head(10))


## 6. Train / Validation / Test Split

We split the combined dataset by the shared emotion label so each split keeps a reasonable class balance.

Then we repeat Arabic rows inside the training split only so the model sees more Arabic during learning.


In [ ]:
train_df, temp_df = train_test_split(
    combined_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=combined_df['label_name'],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=temp_df['label_name'],
)

if ARABIC_TRAIN_REPEAT_FACTOR > 1:
    arabic_train_df = train_df[train_df['language'] == 'ar'].copy()
    train_df = pd.concat([train_df] + [arabic_train_df] * (ARABIC_TRAIN_REPEAT_FACTOR - 1), ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print('Train rows:', len(train_df))
print('Validation rows:', len(val_df))
print('Test rows:', len(test_df))

print('Train language distribution:')
display(table_counts(train_df['language']))
print('Validation language distribution:')
display(table_counts(val_df['language']))
print('Test language distribution:')
display(table_counts(test_df['language']))


## 7. Tokenizer and Model Setup

The tokenizer converts text into integer tokens that the BERT model can process.
The model head is replaced so the classifier outputs exactly the seven shared emotion labels.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

config = AutoConfig.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=len(SHARED_EMOTION_LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True,
)
model.config.problem_type = 'single_label_classification'
model.to(DEVICE)

print('Tokenizer loaded.')
print('Model loaded.')
print('Model device:', DEVICE)
print('Model head adapted for shared 7-way emotion classification.')


## 8. Convert DataFrames to Hugging Face Datasets

Trainer works best with Hugging Face datasets.
We keep only the text and labels columns for training.


In [ ]:
train_hf = Dataset.from_pandas(train_df[['text', 'label_id']].rename(columns={'label_id': 'labels'}), preserve_index=False)
val_hf = Dataset.from_pandas(val_df[['text', 'label_id']].rename(columns={'label_id': 'labels'}), preserve_index=False)
test_hf = Dataset.from_pandas(test_df[['text', 'label_id']].rename(columns={'label_id': 'labels'}), preserve_index=False)


def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)


tokenized_train = train_hf.map(tokenize_batch, batched=True, remove_columns=['text'])
tokenized_val = val_hf.map(tokenize_batch, batched=True, remove_columns=['text'])
tokenized_test = test_hf.map(tokenize_batch, batched=True, remove_columns=['text'])

print(tokenized_train)
print(tokenized_val)
print(tokenized_test)


## 9. Metrics and Class Weights

We use macro-averaged metrics because the label distribution is still uneven even after harmonization.
Macro metrics tell us whether the model works across the full label space, not only on the most frequent classes.

We also compute class weights so the training loop pays more attention to minority labels.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='macro',
        zero_division=0,
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='weighted',
        zero_division=0,
    )
    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
    }


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Inverse-frequency class weights help the model avoid collapsing into the dominant labels.
label_counts = train_df['label_id'].value_counts().reindex(range(len(SHARED_EMOTION_LABELS)), fill_value=1).sort_index()
class_weight_values = label_counts.sum() / (len(SHARED_EMOTION_LABELS) * label_counts.to_numpy())
class_weight_values = class_weight_values / class_weight_values.mean()
CLASS_WEIGHTS = torch.tensor(class_weight_values, dtype=torch.float32)

weights_df = pd.DataFrame({
    'label': SHARED_EMOTION_LABELS,
    'count': label_counts.to_numpy(),
    'class_weight': CLASS_WEIGHTS.numpy(),
})

print('Class weights used by the weighted loss:')
display(weights_df)
print('Metric helper ready.')


## 10. Training Setup

We use a version-aware `TrainingArguments` setup because newer Transformers releases renamed `evaluation_strategy` to `eval_strategy`.

We also use a custom trainer with weighted loss. That usually improves minority-class performance and is one reason this run should take longer than the earlier 20-minute baseline.


In [ ]:
# Build TrainingArguments in a version-aware way because Transformers renamed evaluation_strategy in newer releases.
training_args_kwargs = {
    'output_dir': str(OUTPUT_DIR_PATH),
    'num_train_epochs': NUM_EPOCHS,
    'per_device_train_batch_size': BATCH_SIZE,
    'per_device_eval_batch_size': EVAL_BATCH_SIZE,
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'save_strategy': 'epoch',
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1_macro',
    'greater_is_better': True,
    'logging_steps': 50,
    'dataloader_num_workers': 2,
    'dataloader_pin_memory': torch.cuda.is_available(),
    'fp16': torch.cuda.is_available(),
    'report_to': 'none',
    'seed': RANDOM_SEED,
}

# Newer Transformers versions use eval_strategy; older ones use evaluation_strategy.
training_args_params = TrainingArguments.__init__.__code__.co_varnames
if 'eval_strategy' in training_args_params:
    training_args_kwargs['eval_strategy'] = 'epoch'
elif 'evaluation_strategy' in training_args_params:
    training_args_kwargs['evaluation_strategy'] = 'epoch'

training_args = TrainingArguments(**training_args_kwargs)


class WeightedTrainer(Trainer):
    # Cross-entropy with class weights gives minority labels a stronger training signal.
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(logits.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


# Trainer API changed across Transformers versions.
# Newer releases prefer processing_class=tokenizer; older ones still accept tokenizer=tokenizer.
trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': tokenized_train,
    'eval_dataset': tokenized_val,
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
}

trainer_init_params = Trainer.__init__.__code__.co_varnames
if 'processing_class' in trainer_init_params:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = WeightedTrainer(**trainer_kwargs)

print('Trainer is ready.')
print(training_args)


## 11. Train the Model

This is the main fine-tuning step.

With the new settings, this should take noticeably longer than the earlier 20-minute run on T4 because the notebook now trains for more epochs with smaller batches and weighted loss.


In [ ]:

print('Starting training now...')
train_result = trainer.train()
print('Training finished.')
print(train_result)


## 12. Evaluate the Model

We evaluate on the validation and test splits.

Expected output:

- validation metrics
- test metrics
- classification report
- language-level breakdown
- confusion matrix


In [ ]:
val_metrics = trainer.evaluate(tokenized_val)
print('Validation metrics:')
print(val_metrics)

# Test predictions.
test_output = trainer.predict(tokenized_test)
test_logits = test_output.predictions
test_labels = test_output.label_ids
test_pred_ids = np.argmax(test_logits, axis=-1)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels,
    test_pred_ids,
    average='macro',
    zero_division=0,
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    test_labels,
    test_pred_ids,
    average='weighted',
    zero_division=0,
)
accuracy = accuracy_score(test_labels, test_pred_ids)

print('Test metrics:')
print({
    'accuracy': accuracy,
    'precision_macro': precision,
    'recall_macro': recall,
    'f1_macro': f1,
    'precision_weighted': precision_weighted,
    'recall_weighted': recall_weighted,
    'f1_weighted': f1_weighted,
})

print('Classification report:')
print(classification_report(test_labels, test_pred_ids, target_names=SHARED_EMOTION_LABELS, zero_division=0))

# Add predictions back to the original raw test dataframe for language-level analysis.
test_eval_df = test_df.copy().reset_index(drop=True)
test_eval_df['pred_label_id'] = test_pred_ids
test_eval_df['pred_label_name'] = test_eval_df['pred_label_id'].map(ID2LABEL)
test_eval_df['correct'] = test_eval_df['label_id'] == test_eval_df['pred_label_id']


def summarize_language_metrics(frame, language_code):
    subset = frame[frame['language'] == language_code]
    if len(subset) == 0:
        return {
            'language': language_code,
            'count': 0,
            'accuracy': None,
            'precision_macro': None,
            'recall_macro': None,
            'f1_macro': None,
        }

    precision, recall, f1, _ = precision_recall_fscore_support(
        subset['label_id'],
        subset['pred_label_id'],
        average='macro',
        zero_division=0,
    )
    return {
        'language': language_code,
        'count': len(subset),
        'accuracy': accuracy_score(subset['label_id'], subset['pred_label_id']),
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
    }

language_summary = pd.DataFrame([
    summarize_language_metrics(test_eval_df, 'en'),
    summarize_language_metrics(test_eval_df, 'ar'),
])

print('Language-level test metrics:')
display(language_summary)

# Confusion matrix for the test set.
cm = confusion_matrix(test_labels, test_pred_ids, labels=list(range(len(SHARED_EMOTION_LABELS))))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap='Blues', xticklabels=SHARED_EMOTION_LABELS, yticklabels=SHARED_EMOTION_LABELS)
plt.title('Test Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 13. Example Inference

After training, we test a few sentences in both English and Arabic.

The model returns:

- `sentiment`
- `confidence`
- `primary_emotion`
- `emotion_scores`


In [ ]:
EMOTION_TO_SENTIMENT = {
    'joy': 'positive',
    'sadness': 'negative',
    'anger': 'negative',
    'fear': 'negative',
    'disgust': 'negative',
    'surprise': 'neutral',
    'neutral': 'neutral',
}


def predict_emotion(text):
    model.eval()
    cleaned_text = clean_text(text)
    inputs = tokenizer(
        cleaned_text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    top_index = int(np.argmax(probs))
    top_label = ID2LABEL[top_index]
    top_sentiment = EMOTION_TO_SENTIMENT.get(top_label, 'neutral')

    top_indices = np.argsort(probs)[::-1][:5]
    top_emotions = [
        {'label': ID2LABEL[int(idx)], 'score': float(probs[int(idx)])}
        for idx in top_indices
    ]

    return {
        'sentiment': top_sentiment,
        'confidence': float(probs[top_index]),
        'primary_emotion': top_label,
        'emotion_scores': top_emotions,
    }

sample_texts = [
    'I am really happy with this result!',
    'هذا الخبر جميل جدًا وأنا سعيد به',
    'I am frustrated and disappointed by the delay.',
    'أنا غاضب جدًا من هذا التصرف',
    'أنا خائف من ما سيحدث بعد ذلك',
]

for sample in sample_texts:
    result = predict_emotion(sample)
    print('Text:', sample)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print('-')


## 14. Save the Model

We save the final model, tokenizer, and metadata into the local project folder first.

The same files can then be uploaded to Google Drive and later loaded by `nlp/trained_bert_model.py`.


In [ ]:
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR_PATH))
tokenizer.save_pretrained(OUTPUT_DIR_PATH)

metadata = {
    'model_name': 'fine_tuned_goemotions_arpanemo_bert',
    'base_model': BASE_MODEL_NAME,
    'english_dataset': ENGLISH_DATASET_NAME,
    'arabic_dataset': ARABIC_DATASET_NAME,
    'label_space': SHARED_EMOTION_LABELS,
    'label_mapping_notes': {
        'english_goemotions_to_shared': 'GoEmotions fine-grained labels are collapsed into a shared 7-way emotion space',
        'arabic_happiness_to': 'joy',
    },
    'training_config': {
        'max_length': MAX_LENGTH,
        'max_english_samples': MAX_ENGLISH_SAMPLES,
        'arabic_train_repeat_factor': ARABIC_TRAIN_REPEAT_FACTOR,
        'batch_size': BATCH_SIZE,
        'eval_batch_size': EVAL_BATCH_SIZE,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
    },
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'output_format': {
        'sentiment': 'positive | negative | neutral',
        'confidence': 'max softmax probability',
        'primary_emotion': 'predicted shared emotion label',
        'emotion_scores': 'top emotion probabilities',
    },
}

metadata_path = OUTPUT_DIR_PATH / 'model_metadata.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Model saved to:', OUTPUT_DIR_PATH.resolve())
print('Tokenizer saved to:', OUTPUT_DIR_PATH.resolve())
print('Metadata saved to:', metadata_path.resolve())
print('The saved model can later be loaded from nlp/trained_bert_model.py or another inference module.')


## 15. Upload the Model to Google Drive

This cell mounts Google Drive in Colab and copies the final model files there.

You can change the target folder if you want to keep multiple experiments side by side.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

drive_target_dir = Path('/content/drive/MyDrive/fine_tuned_bert')
drive_target_dir.mkdir(parents=True, exist_ok=True)

for item in OUTPUT_DIR_PATH.iterdir():
    if item.is_file():
        shutil.copy2(item, drive_target_dir / item.name)

print('Uploaded model files to:', drive_target_dir)
print('Files uploaded:')
for item in sorted(drive_target_dir.iterdir()):
    if item.is_file():
        print('-', item.name)


## 16. Final Summary

What this notebook trained:

- a multilingual BERT emotion classifier
- on English GoEmotions and Arabic ArPanEmo
- using one shared 7-way emotion label space

What changed compared with the earlier version:

- we stopped forcing the data into 28 fine-grained labels
- we used a stronger multilingual BERT base encoder
- we added class weights to reduce majority-class bias
- we increased the training time so the run is more meaningful on a T4

Where the model is saved:

- local project path: `models/fine_tuned_bert/`
- Google Drive path in Colab: `/content/drive/MyDrive/fine_tuned_bert/`

How it will connect later:

- `nlp/trained_bert_model.py` can load the saved folder
- `nlp/sentiment_service.py` can use the trained model provider later
- the output stays compatible with the current project shape: sentiment, confidence, primary_emotion, emotion_scores, model_name, model_version

Main limitation of this first bilingual version:

- GoEmotions is naturally multi-label, but this notebook still turns it into a single shared emotion label per row for simplicity and easier integration.
- That is a good first production step, but a true multi-label version would be the next upgrade if we want even better emotion coverage.
